**Data exploration**

In [0]:
spark.sql("SHOW TABLES IN bq_raw_statsbomb_sa_catalog.raw_statsbomb").show(truncate=False)
# Databricks notebook source
full_name = "bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches"
df = spark.table(full_name)

display(df.limit(10))
print("rows (approx / action may scan):", df.count())
df.printSchema()
spark.sql("""
          SELECT 'matches' AS table_name, COUNT(*) AS n_rows
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches
UNION ALL
SELECT 'lineups', COUNT(*)
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.lineups
UNION ALL
SELECT 'events', COUNT(*)
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events;
"""
).show()
spark.sql("""
SELECT 'matches' as table_name, COUNT(*) as row_count FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches
UNION ALL
SELECT 'events', COUNT(*) FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events
UNION ALL
SELECT 'lineups', COUNT(*) FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.lineups;
"""
).show()

**Rows count per competition/season**

In [0]:
%sql
SELECT competition_name, season_name, COUNT(*) AS n_matches
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches
GROUP BY competition_name, season_name
ORDER BY competition_name, season_name;

In [0]:
%sql
SELECT m.competition_name, m.season_name, COUNT(*) AS n_events
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events e
JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches m
  ON e.match_id = m.match_id
GROUP BY m.competition_name, m.season_name
ORDER BY m.competition_name, m.season_name;

In [0]:
%sql
SELECT m.competition_name, m.season_name, COUNT(*) AS n_lineup_rows
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.lineups l
JOIN bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches m
  ON l.match_id = m.match_id
GROUP BY m.competition_name, m.season_name
ORDER BY m.competition_name, m.season_name;

**Event type distribution**

In [0]:
%sql
SELECT type, COUNT(*) AS n, ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events
GROUP BY type
ORDER BY n DESC;

**xG distribution (shots only)**

In [0]:
%sql
SELECT
  COUNT(*) AS n_shots_with_xg,
  ROUND(AVG(shot_statsbomb_xg), 4) AS mean_xg,
  ROUND(STDDEV(shot_statsbomb_xg), 4) AS sd_xg,
  MIN(shot_statsbomb_xg) AS min_xg,
  MAX(shot_statsbomb_xg) AS max_xg
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events
WHERE type = 'Shot' AND shot_statsbomb_xg IS NOT NULL;

In [0]:
%sql
SELECT
  WIDTH_BUCKET(shot_statsbomb_xg, 0, 1.0, 20) AS bin_0_to_1,
  COUNT(*) AS n
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.events
WHERE type = 'Shot' AND shot_statsbomb_xg IS NOT NULL AND shot_statsbomb_xg >= 0 AND shot_statsbomb_xg <= 1
GROUP BY bin_0_to_1
ORDER BY bin_0_to_1;

**Home / away / draw rates**

Overall

In [0]:
%sql
SELECT
  SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS n_home_win,
  SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) AS n_away_win,
  SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) AS n_draw,
  COUNT(*) AS n_total,
  ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_home,
  ROUND(100.0 * SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_away,
  ROUND(100.0 * SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_draw
FROM bq_raw_statsbomb_sa_catalog.raw_statsbomb.matches
WHERE home_score IS NOT NULL AND away_score IS NOT NULL;

By competition

In [0]:
RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

spark.sql(f"""
SELECT
  SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS n_home_win,
  SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) AS n_away_win,
  SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) AS n_draw,
  COUNT(*) AS n_total,
  ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_home,
  ROUND(100.0 * SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_away,
  ROUND(100.0 * SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_draw
FROM {RAW}.matches
WHERE home_score IS NOT NULL AND away_score IS NOT NULL
""").show(truncate=False)

By competition and season

In [0]:
spark.sql(f"""
SELECT
  competition_name,
  SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS n_home_win,
  SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) AS n_away_win,
  SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) AS n_draw,
  COUNT(*) AS n_matches,
  ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_home,
  ROUND(100.0 * SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_away,
  ROUND(100.0 * SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_draw
FROM {RAW}.matches
WHERE home_score IS NOT NULL AND away_score IS NOT NULL
GROUP BY competition_name
ORDER BY n_matches DESC
""").show(truncate=False)

In [0]:
spark.sql(f"""
SELECT
  competition_name,
  season_name,
  SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) AS n_home_win,
  SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) AS n_away_win,
  SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) AS n_draw,
  COUNT(*) AS n_matches,
  ROUND(100.0 * SUM(CASE WHEN home_score > away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_home,
  ROUND(100.0 * SUM(CASE WHEN home_score < away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_away,
  ROUND(100.0 * SUM(CASE WHEN home_score = away_score THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_draw
FROM {RAW}.matches
WHERE home_score IS NOT NULL AND away_score IS NOT NULL
GROUP BY competition_name, season_name
ORDER BY competition_name, season_name
""").show(truncate=False)

**Player minutes distribution**

In [0]:
import re
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

def parse_time_to_minutes(t):
    if t is None:
        return None
    s = str(t).strip()
    if s == "" or s.lower() == "nan":
        return None
    if "+" in s:
        base, rest = s.split("+", 1)
        try:
            extra = float(rest.split(":")[0])
        except ValueError:
            extra = 0.0
        try:
            return 90.0 + extra + float(rest.split(":")[1]) / 60.0 if ":" in rest else 90.0 + extra
        except (ValueError, IndexError):
            return 90.0 + extra
    parts = s.split(":")
    if len(parts) != 2:
        return None
    try:
        return float(parts[0]) + float(parts[1]) / 60.0
    except ValueError:
        return None

parse_udf = F.udf(parse_time_to_minutes, DoubleType())

lineups = spark.table(f"{RAW}.lineups")

minutes_df = (
    lineups
    .withColumn("from_min", parse_udf(F.col("from_time")))
    .withColumn("to_min", parse_udf(F.col("to_time")))
    .withColumn(
        "minutes_played",
        F.when(F.col("from_min").isNotNull() & F.col("to_min").isNotNull(),
               F.least(F.lit(120.0), F.greatest(F.lit(0.0), F.col("to_min") - F.col("from_min"))))
    )
)

print("Per lineup row — minutes_played summary:")
minutes_df.select("minutes_played").summary().show()

print("Histogram bins (0–15, 15–30, …, 105–120):")
minutes_df = minutes_df.withColumn(
    "min_bin",
    F.when(F.col("minutes_played").isNull(), F.lit("null"))
     .when(F.col("minutes_played") < 15, "0-15")
     .when(F.col("minutes_played") < 30, "15-30")
     .when(F.col("minutes_played") < 45, "30-45")
     .when(F.col("minutes_played") < 60, "45-60")
     .when(F.col("minutes_played") < 75, "60-75")
     .when(F.col("minutes_played") < 90, "75-90")
     .when(F.col("minutes_played") <= 105, "90-105")
     .otherwise("105-120")
)
minutes_df.groupBy("min_bin").count().orderBy("min_bin").show(truncate=False)

# Optional: total minutes per player (entire table scan)
player_totals = (
    minutes_df.filter(F.col("minutes_played").isNotNull())
    .groupBy("player_name")
    .agg(F.sum("minutes_played").alias("total_minutes"))
)
print("Per player total minutes — quantiles (approx):")
qs = player_totals.approxQuantile("total_minutes", [0.0, 0.25, 0.5, 0.75, 0.9, 0.99, 1.0], 0.01)
print(dict(zip(["min", "p25", "p50", "p75", "p90", "p99", "max"], qs)))

**6. Data quality flags (SQL)**

In [0]:
RAW = "bq_raw_statsbomb_sa_catalog.raw_statsbomb"

spark.sql(f"""
SELECT 'events: match_id NULL' AS check_name, COUNT(*) AS n
FROM {RAW}.events WHERE match_id IS NULL
UNION ALL
SELECT 'lineups: match_id NULL', COUNT(*)
FROM {RAW}.lineups WHERE match_id IS NULL
UNION ALL
SELECT 'events orphan match_id', COUNT(*)
FROM {RAW}.events e
LEFT ANTI JOIN {RAW}.matches m ON e.match_id = m.match_id
UNION ALL
SELECT 'lineups orphan match_id', COUNT(*)
FROM {RAW}.lineups l
LEFT ANTI JOIN {RAW}.matches m ON l.match_id = m.match_id
UNION ALL
SELECT 'shots with NULL xg', COUNT(*)
FROM {RAW}.events WHERE type = 'Shot' AND shot_statsbomb_xg IS NULL
UNION ALL
SELECT 'matches NULL home_score', COUNT(*)
FROM {RAW}.matches WHERE home_score IS NULL
UNION ALL
SELECT 'matches NULL away_score', COUNT(*)
FROM {RAW}.matches WHERE away_score IS NULL
UNION ALL
SELECT 'lineups NULL from_time', COUNT(*)
FROM {RAW}.lineups WHERE from_time IS NULL
UNION ALL
SELECT 'lineups NULL to_time', COUNT(*)
FROM {RAW}.lineups WHERE to_time IS NULL
UNION ALL
SELECT 'lineups NULL player_name', COUNT(*)
FROM {RAW}.lineups WHERE player_name IS NULL
UNION ALL
SELECT 'lineups NULL position_name', COUNT(*)
FROM {RAW}.lineups WHERE position_name IS NULL
""").show(truncate=False)